In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    current_timestamp, current_date, lit, sha2, concat_ws,
    col, trim, to_json, struct
)
import uuid

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "bronze"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "S3-Retail-Lakehouse-Bucket"
base_path = "/Volumes/retail/default/raw_data"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"
ingested_files_table = "retail.audit.ingested_files"

batch_id = f"batch_incremental_{str(uuid.uuid4())[:8]}"

table_configs = [
    {
        "table_name": "ns_brands",
        "target_table": "retail.bronze.ns_brands",
        "business_key": "internal_id",
        "cast_cols": ["internal_id", "brand_code", "brand_name", "is_active", "created_at"],
        "hash_cols": ["internal_id", "brand_code", "brand_name", "is_active", "created_at"],
        "required_cols": ["internal_id", "brand_code", "brand_name", "is_active", "created_at"]
    },
    {
        "table_name": "ns_customers",
        "target_table": "retail.bronze.ns_customers",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "entity_id", "company_name", "customer_type", "email", "phone",
            "subsidiary", "currency_code", "terms_name", "vat_number",
            "billing_address1", "billing_city", "billing_state", "billing_country",
            "shipping_address1", "shipping_city", "shipping_state", "shipping_country",
            "is_active", "created_at"
        ],
        "hash_cols": [
            "internal_id", "entity_id", "company_name", "customer_type", "email", "phone",
            "subsidiary", "currency_code", "terms_name", "vat_number",
            "billing_address1", "billing_city", "billing_state", "billing_country",
            "shipping_address1", "shipping_city", "shipping_state", "shipping_country",
            "is_active", "created_at"
        ],
        "required_cols": [
            "internal_id", "entity_id", "company_name", "customer_type",
            "subsidiary", "currency_code", "terms_name", "is_active", "created_at"
        ]
    },
    {
        "table_name": "ns_items",
        "target_table": "retail.bronze.ns_items",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "item_id", "item_name", "item_type", "category",
            "subcategory", "brand_internal_id", "base_price", "cost_price",
            "is_active", "created_at"
        ],
        "hash_cols": [
            "internal_id", "item_id", "item_name", "item_type", "category",
            "subcategory", "brand_internal_id", "base_price", "cost_price",
            "is_active", "created_at"
        ],
        "required_cols": [
            "internal_id", "item_id", "item_name", "item_type",
            "category", "subcategory", "base_price", "cost_price",
            "is_active", "created_at"
        ]
    },
    {
        "table_name": "ns_sales_orders",
        "target_table": "retail.bronze.ns_sales_orders",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date",
            "order_status", "sales_channel", "payment_status", "currency_code",
            "external_ref_num", "integration_source", "order_total", "created_at"
        ],
        "hash_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date",
            "order_status", "sales_channel", "payment_status", "currency_code",
            "external_ref_num", "integration_source", "order_total", "created_at"
        ],
        "required_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date",
            "order_status", "sales_channel", "payment_status", "currency_code",
            "integration_source", "order_total", "created_at"
        ]
    },
    {
        "table_name": "ns_sales_order_lines",
        "target_table": "retail.bronze.ns_sales_order_lines",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "sales_order_internal_id", "line_num",
            "item_internal_id", "quantity", "unit_rate",
            "line_amount", "created_at"
        ],
        "hash_cols": [
            "internal_id", "sales_order_internal_id", "line_num",
            "item_internal_id", "quantity", "unit_rate",
            "line_amount", "created_at"
        ],
        "required_cols": [
            "internal_id", "sales_order_internal_id", "line_num",
            "item_internal_id", "quantity", "unit_rate",
            "line_amount", "created_at"
        ]
    },
    {
        "table_name": "ns_shipments",
        "target_table": "retail.bronze.ns_shipments",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id",
            "warehouse_name", "shipment_date", "delivery_date",
            "shipment_status", "tracking_number", "created_at"
        ],
        "hash_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id",
            "warehouse_name", "shipment_date", "delivery_date",
            "shipment_status", "tracking_number", "created_at"
        ],
        "required_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id",
            "warehouse_name", "shipment_date", "shipment_status", "created_at"
        ]
    }
]

# =========================================================
# AUDIT HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(
    run_id,
    rows_read,
    rows_written,
    rows_rejected,
    records_before,
    records_after
):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = CAST({rows_read} AS BIGINT),
          rows_written = CAST({rows_written} AS BIGINT),
          rows_rejected = CAST({rows_rejected} AS BIGINT),
          records_before = CAST({records_before} AS BIGINT),
          records_after = CAST({records_after} AS BIGINT)
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = str(error_message).replace("'", "''")[:5000]

    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


# =========================================================
# MAIN BRONZE INCREMENTAL LOAD
# =========================================================
def load_incremental_to_bronze(cfg):
    table_name = cfg["table_name"]
    target_table = cfg["target_table"]
    business_key = cfg["business_key"]

    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        records_before = spark.table(target_table).count()

        # 1. List all parquet files in landing volume
        files = dbutils.fs.ls(base_path)

        # 3. Remove already successfully ingested files
        df_files = (
            spark.createDataFrame(files)
            .select(
                col("path").alias("source_file_path"),
                F.element_at(F.split(col("path"), "/"), -1).alias("source_file_name")
            )
            .filter(col("source_file_path").endswith(".parquet"))
        )

        df_files = df_files.filter(
            col("source_file_name").rlike(f"^{table_name}_[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}\\.parquet$")
        )

        df_ingested = (
            spark.table(ingested_files_table)
            .filter(
                (col("table_name") == table_name) &
                (col("ingestion_status") == "SUCCESS")
            )
            .select("source_file_name")
            .dropDuplicates()
        )

        df_new_files = df_files.join(
            df_ingested,
            on="source_file_name",
            how="left_anti"
        )

        new_files = [r.source_file_path for r in df_new_files.collect()]

        if not new_files:
            end_audit_success(
                run_id=run_id,
                rows_read=0,
                rows_written=0,
                rows_rejected=0,
                records_before=records_before,
                records_after=records_before
            )

            print(f"{target_table}: no new files found")
            return

        print(f"{table_name}: new files found:")
        for f in new_files:
            print(f"  - {f}")

        # 4. Read only new files
        df = spark.read.parquet(*new_files)

        rows_read = df.count()

        # 5. Cast Bronze business columns to string
        for c in cfg["cast_cols"]:
            df = df.withColumn(c, col(c).cast("string"))

        # 6. Reject required-column missing/blank rows
        reject_condition = None

        for c in cfg["required_cols"]:
            cond = col(c).isNull() | (trim(col(c)) == "")
            reject_condition = cond if reject_condition is None else (reject_condition | cond)

        df_valid = df.filter(~reject_condition)
        df_rejects = df.filter(reject_condition)

        rows_rejected = df_rejects.count()

        # 7. Log rejected rows, if any
        if rows_rejected > 0:
            df_rejects_log = (
                df_rejects
                .withColumn("source_record_id", col(business_key).cast("string"))
                .withColumn("reject_stage", lit("BRONZE"))
                .withColumn("reject_reason", lit("MISSING_REQUIRED_VALUE"))
                .withColumn("reject_column", lit(",".join(cfg["required_cols"])))
                .withColumn("severity", lit("ERROR"))
                .withColumn("raw_payload", to_json(struct(*[col(c) for c in df_rejects.columns])))
                .withColumn("error_code", lit("BRZ_001"))
                .withColumn("error_message", lit("One or more required columns are null or blank"))
                .withColumn("reject_id", F.expr("uuid()"))
                .withColumn("run_id", lit(run_id))
                .withColumn("batch_id", lit(batch_id))
                .withColumn("pipeline_name", lit(pipeline_name))
                .withColumn("layer_name", lit(layer_name))
                .withColumn("table_name", lit(table_name))
                .withColumn("source_system", lit(source_system))
                .withColumn("rejected_at", current_timestamp())
                .withColumn("created_at", current_timestamp())
                .select(
                    "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
                    "table_name", "source_system", "source_record_id", "reject_stage",
                    "reject_reason", "reject_column", "severity", "raw_payload",
                    "error_code", "error_message", "rejected_at", "created_at"
                )
            )

            df_rejects_log.write.format("delta").mode("append").saveAsTable(reject_table)

        # 8. Add Bronze metadata
        df_final = (
            df_valid
            .withColumn("etl_loaded_at", current_timestamp().cast("string"))
            .withColumn("source_system", lit(source_system))
            .withColumn("etl_run_id", lit(run_id))
            .withColumn("ingestion_date", current_date())
            .withColumn(
                "record_hash",
                sha2(
                    concat_ws(
                        "||",
                        *[
                            F.coalesce(col(c).cast("string"), lit(""))
                            for c in cfg["hash_cols"]
                        ]
                    ),
                    256
                )
            )
            .withColumn("is_deleted", lit("false"))
        )

        # 9. Match target Bronze table columns
        target_cols = spark.table(target_table).columns
        df_final = df_final.select(*target_cols)

        rows_written = df_final.count()

        # 10. Append to Bronze table
        df_final.write.format("delta").mode("append").saveAsTable(target_table)

        records_after = spark.table(target_table).count()

        # 11. Log newly ingested files
        (
            df_new_files
            .withColumn(
                "source_file_name",
                F.element_at(F.split(col("source_file_path"), "/"), -1)
            )
            .withColumn("file_id", F.expr("uuid()"))
            .withColumn("run_id", lit(run_id))
            .withColumn("batch_id", lit(batch_id))
            .withColumn("pipeline_name", lit(pipeline_name))
            .withColumn("layer_name", lit(layer_name))
            .withColumn("table_name", lit(table_name))
            .withColumn("source_system", lit(source_system))
            .withColumn("ingestion_status", lit("SUCCESS"))
            .withColumn("rows_read", lit(rows_read).cast("bigint"))
            .withColumn("rows_written", lit(rows_written).cast("bigint"))
            .withColumn("ingested_at", current_timestamp())
            .select(
                "file_id", "run_id", "batch_id", "pipeline_name", "layer_name",
                "table_name", "source_system", "source_file_path", "source_file_name",
                "ingestion_status", "rows_read", "rows_written", "ingested_at"
            )
            .write.format("delta")
            .mode("append")
            .saveAsTable(ingested_files_table)
        )

        # 12. Finalize audit
        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        print(f"{target_table}: {rows_written} incremental rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# RUN ALL TABLES
# =========================================================
for cfg in table_configs:
    load_incremental_to_bronze(cfg)

In [0]:
base_path = "/Volumes/retail/default/raw_data"
files = dbutils.fs.ls(base_path)
files_df = spark.createDataFrame(files)

files_df = files_df.filter(col("path").endswith(".parquet"))
files_df = files_df.withColumnRenamed("path", "source_file_path")